# Experiments 5 · Formal Open-world Generalization

目标：验证 PartNet 上的 representation 结论能否推广到 **free-form natural language**。

本 notebook **只复用旧 open-world 数据里的 description / category / object_id**。旧的 `part_spec`、`anchor_tree`、`global_tree`、`hierarchy_local_tree` 全部忽略并重新生成。

顺序运行 Cell 1–10。Cell 4–5 会调用 LLM；所有请求均断点缓存。Cell 6 之后不再调用 API。


## Cell 1 — 导入


In [13]:
import os
import importlib
import pandas as pd
import methods
from IPython.display import display
importlib.reload(methods)
from methods import *


## Cell 2 — 配置

只需要重点修改 `openworld_source_path`：指向你旧 experiments1 的 open-world JSON。文件可以包含旧生成结果；loader 只提取 description/category/object_id。

**不要把 output_dir 指向旧 experiments1 结果目录。**


In [7]:
cfg = PartNetConfig()
cfg.project_dir = "."
cfg.output_dir = "openworld_results"
cfg.model = os.getenv("DEEPSEEK_MODEL", "deepseek-chat")
cfg.temperature = 0.0
cfg.max_tokens = 8192
cfg.max_attempts = 3
cfg.workers = 4
cfg.allow_api = True
cfg.expected_objects = None
cfg.object_limit = None
cfg.random_anchor_seeds = (0, 1, 2, 3, 4)
cfg.scale_factors = (0.8, 1.2, 1.5)
cfg.bootstrap_samples = 10000
cfg.statistics_seed = 42

openworld_source_path="openworld_objects_120.json"

## Cell 3 — 冻结自由语言输入


In [8]:
inputs = load_openworld_inputs(openworld_source_path, cfg)
display(pd.DataFrame(inputs["objects"])[["object_id", "category", "description"]].head(10))
print("objects:", len(inputs["objects"]))


Open-world descriptions: 120 objects; frozen manifest=D:\MyJupyter\Works\ASPT\openworld_results\openworld_manifest.json


,object_id,category,description
0,OW001,Furniture,A round café table with a single circular top ...
1,OW002,Furniture,A small wooden stool with a square seat and fo...
2,OW003,Furniture,A wall shelf made from one horizontal board he...
3,OW004,Furniture,A basic bench with one long seat plank support...
4,OW005,Furniture,"A narrow side table with a rectangular top, fo..."
5,OW006,Furniture,"An office chair with a padded seat, a separate..."
6,OW007,Furniture,"A dining chair with a flat seat, four legs, an..."
7,OW008,Furniture,"A bedside cabinet with a rectangular body, one..."
8,OW009,Furniture,A coffee table with an oval top supported by t...
9,OW010,Furniture,"A writing desk with a wide tabletop, two drawe..."


objects: 120


## Cell 4 — Free-text → shared parts + connections

这是 open-world 和 PartNet 最大的区别：这里 **没有 GT part inventory**。LLM 从自由文本推断一次 shared structure，之后三个 representation 完全共用。这里只对 JSON/tree schema 做 retry，不使用任何 GT 修正 semantic decomposition。


In [9]:
shared = run_openworld_shared_specs(inputs, cfg)
shared_rows = list(shared.values())
print("shared valid:", sum(r["status"] == "valid" for r in shared_rows), "/", len(shared_rows))
print("first-attempt valid:", sum(r.get("first_attempt_valid", False) for r in shared_rows))
for row in shared_rows:
    if row["status"] != "valid":
        print("FAILED SHARED:", row["object_id"], row["category"], row["status"], row.get("attempt_count"))


Open-world shared decomposition: 20/120
Open-world shared decomposition: 40/120
Open-world shared decomposition: 60/120
Open-world shared decomposition: 80/120
Open-world shared decomposition: 100/120
Open-world shared decomposition: 120/120
Open-world shared specs: valid=119/120
shared valid: 119 / 120
first-attempt valid: 101
FAILED SHARED: OW107 MechanicalComposite exhausted 3


## Cell 5 — Anchor / Global / Hierarchy Local 公平生成

三个分支使用相同 description、parts、part IDs、connections、model、temperature 和 retry policy。


In [10]:
generation = run_openworld_generation(shared, cfg)
for method in REPRESENTATIONS:
    rows = generation[method]
    print(method, "valid=", sum(r.get("status") == "valid" for r in rows.values()), "/", len(rows))

cfg.allow_api = False


anchor: 20/119
anchor: 40/119
anchor: 60/119
anchor: 80/119
anchor: 100/119
anchor: 119/119
anchor: valid=119/120
global: 20/119
global: 40/119
global: 60/119
global: 80/119
global: 100/119
global: 119/119
global: valid=119/120
local: 20/119
local: 40/119
local: 60/119
local: 80/119
local: 100/119
local: 119/119
local: valid=119/120
anchor valid= 119 / 119
global valid= 119 / 119
local valid= 119 / 119


## Cell 6 — 无 GT 的静态结构评价


In [14]:
evaluation = run_openworld_evaluation(inputs, shared, generation, cfg)
print("complete cases:", len(evaluation["complete_ids"]), "/", len(inputs["objects"]))
static_rows = evaluation["objects"]
summary_preview = methods._summaries(static_rows, ("method",), tuple(OPENWORLD_METRIC_DEFINITIONS))
display(pd.DataFrame(summary_preview).pivot(index="method", columns="metric", values="mean").round(6))


Open-world static evaluation: complete=119/120; evaluation failures=0
complete cases: 119 / 120


metric,obb_overlap_pair_fraction,part_volume_sum,shared_edge_obb_sat_deviation,shared_edge_obb_sat_gap,shared_edge_obb_sat_penetration
method,,,,,
anchor,0.132087,0.105393,0.017910,0.002186,0.015724
global,0.187265,0.129816,0.052927,0.022196,0.030731
local,0.215964,0.128677,0.049353,0.015164,0.034190


## Cell 7 — Random Anchor 消融（不调用 LLM）


In [15]:
random_anchor = run_openworld_random_anchor(inputs, shared, generation, evaluation, cfg)
print("random-anchor objects:", len(random_anchor["objects"]))


Open-world Random Anchor: 119 objects x 5 seeds
random-anchor objects: 119


## Cell 8 — All-shared-edge 尺寸扰动（不调用 LLM）


In [16]:
perturbation = run_partnet_scale_perturbation(inputs, shared, generation, evaluation, cfg)
print("perturbation records:", len(perturbation["objects"]))


All-edge scale perturbation: 1071 method-object-scale records；未调用 LLM。
perturbation records: 1071


## Cell 9 — 正式统计与结果表

这里的自动指标只评价 representation-level structural geometry。**没有 GT 3D instance，所以不要把它们解释成 text fidelity。** Open-world 的语义/视觉正确性下一步再做 human evaluation。


In [19]:
tables = export_openworld_results(inputs, shared, generation, evaluation, random_anchor, perturbation, cfg)
print("STATIC SUMMARY")
display(openworld_summary_view(tables, "summary"))
print("PERTURBATION SUMMARY")
display(openworld_summary_view(tables, "perturbation_summary"))

stats = pd.DataFrame(tables["paired_statistics"])
columns = ["experiment", "baseline", "metric", "scale", "n", "anchor_mean", "baseline_mean", "mean_improvement", "ci_low", "ci_high", "holm_p", "wins", "ties", "losses"]
display(stats[columns])

status = pd.DataFrame(tables["generation_status"])
print("complete:", int(status["complete_case"].sum()), "/", len(status))
display(status.groupby("category")["complete_case"].agg(["count", "sum"]))


Open-world results exported: complete=119/120
STATIC SUMMARY


metric,obb_overlap_pair_fraction,part_volume_sum,shared_edge_obb_sat_deviation,shared_edge_obb_sat_gap,shared_edge_obb_sat_penetration
method,,,,,
anchor,0.132087,0.105393,0.017910,0.002186,0.015724
global,0.187265,0.129816,0.052927,0.022196,0.030731
local,0.215964,0.128677,0.049353,0.015164,0.034190
random_anchor,0.371030,0.105393,0.032040,0.002139,0.029901


PERTURBATION SUMMARY


metric        baseline_contact_deviation  contact_deviation  deviation_change  \
scale method                                                                    
0.8   anchor                    0.017910           0.016199         -0.001711   
      global                    0.052927           0.063131          0.010204   
      local                     0.049353           0.055321          0.005968   
1.2   anchor                    0.017910           0.019290          0.001380   
      global                    0.052927           0.070184          0.017257   
      local                     0.049353           0.064429          0.015076   
1.5   anchor                    0.017910           0.021242          0.003332   
      global                    0.052927           0.092308          0.039381   
      local                     0.049353           0.083394          0.034040   

metric        margin_drift  
scale method                
0.8   anchor      0.001845  
      global      0.022598  
      local       0.018870  
1.2   anchor      0.001514  
      global      0.020104  
      local       0.017439  
1.5   anchor      0.003666  
      global      0.045505  
      local       0.039734

,experiment,baseline,metric,scale,n,anchor_mean,baseline_mean,mean_improvement,ci_low,ci_high,holm_p,wins,ties,losses
0,static,global,shared_edge_obb_sat_deviation,NaN,119,0.017910,0.052927,0.035017,0.025266,0.046052,8.375442e-11,86,21,12
1,static,global,shared_edge_obb_sat_gap,NaN,119,0.002186,0.022196,0.020010,0.012210,0.029365,9.699962e-08,41,78,0
2,static,global,obb_overlap_pair_fraction,NaN,119,0.132087,0.187265,0.055177,0.024898,0.087674,2.637312e-03,55,40,24
3,static,local,shared_edge_obb_sat_deviation,NaN,119,0.017910,0.049353,0.031443,0.023870,0.039433,1.462548e-14,91,18,10
4,static,local,shared_edge_obb_sat_gap,NaN,119,0.002186,0.015164,0.012977,0.008216,0.018218,8.239682e-08,42,77,0
5,static,local,obb_overlap_pair_fraction,NaN,119,0.132087,0.215964,0.083877,0.052613,0.116232,3.161282e-06,64,38,17
6,random_anchor,random_anchor,shared_edge_obb_sat_deviation,NaN,119,0.017910,0.032040,0.014130,0.008444,0.019413,1.453715e-08,96,0,23
7,random_anchor,random_anchor,shared_edge_obb_sat_gap,NaN,119,0.002186,0.002139,-0.000048,-0.000578,0.000522,1.000000e+00,1,116,2
8,random_anchor,random_anchor,obb_overlap_pair_fraction,NaN,119,0.132087,0.371030,0.238943,0.204836,0.273190,1.539446e-16,106,1,12
9,scale_perturbation,global,contact_deviation,0.8,119,0.016199,0.063131,0.046932,0.037928,0.057248,2.195301e-18,113,0,6


complete: 119 / 120


,count,sum
category,,
Appliances,20,20
Electronics,20,20
Furniture,20,20
MechanicalComposite,20,19
Tools,20,20
Vehicles,20,20


## Cell 10 — 打包结果给我分析


In [20]:
from zipfile import ZipFile, ZIP_DEFLATED

review_files = [
    "openworld_manifest.json",
    "shared_specs.json",
    "evaluation_all.json",
    "random_anchor_all.json",
    "perturbation_all.json",
    "tables/results.json",
    "evaluation_protocol.json",
]

review_path = cfg.output_path("experiments5_review.zip")
with ZipFile(review_path, "w", compression=ZIP_DEFLATED) as archive:
    for name in review_files:
        path = cfg.output_path(name)
        if not path.is_file():
            raise FileNotFoundError(path)
        archive.write(path, arcname=name)

print("upload:", review_path.resolve())


upload: D:\MyJupyter\Works\ASPT\openworld_results\experiments5_review.zip
